In [1]:
import sys
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\ml_eng_prep_2026\project-1-secom-yield")
INTERIM = ROOT / "data" / "interim"
sys.path.insert(0, str(ROOT))

from src.load import load_secom
from src.features import build_feature_spec, apply_feature_spec

X, y, ts = load_secom()
assert X.shape == (1567, 590)
print(y.sum())

104


In [2]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    X.index, test_size=0.2, stratify=y, random_state=42,
)

for name, idx in [("train", train_idx), ("test", test_idx)]:
    print(name, len(idx), y[idx].sum(), round(y[idx].mean(), 4))

splits = pd.DataFrame({"idx": X.index})
splits["split"] = "test"
splits.loc[train_idx, "split"] = "train"
splits.to_csv(INTERIM / "split_idx.csv", index=False)

train 1253 83 0.0662
test 314 21 0.0669


In [3]:
X_tr = X.loc[train_idx]

top_share = X_tr.apply(lambda s: s.value_counts(normalize=True).iloc[0])
miss = X_tr.isna().mean()

print((top_share > 0.99).sum(), (miss > 0.45).sum())
print(miss.sort_values(ascending=False).head(40))
print(X_tr["sensor_157"].isna().equals(X_tr["sensor_158"].isna()))

122 32
sensor_292    0.906624
sensor_293    0.906624
sensor_158    0.906624
sensor_157    0.906624
sensor_220    0.856345
sensor_085    0.856345
sensor_358    0.856345
sensor_492    0.856345
sensor_384    0.663208
sensor_382    0.663208
sensor_383    0.663208
sensor_245    0.663208
sensor_109    0.663208
sensor_244    0.663208
sensor_516    0.663208
sensor_246    0.663208
sensor_110    0.663208
sensor_517    0.663208
sensor_111    0.663208
sensor_518    0.663208
sensor_578    0.612929
sensor_580    0.612929
sensor_581    0.612929
sensor_579    0.612929
sensor_346    0.490822
sensor_072    0.490822
sensor_073    0.490822
sensor_345    0.490822
sensor_247    0.453312
sensor_385    0.453312
sensor_519    0.453312
sensor_112    0.453312
sensor_563    0.185954
sensor_564    0.185954
sensor_566    0.185954
sensor_562    0.185954
sensor_568    0.185954
sensor_569    0.185954
sensor_567    0.185954
sensor_565    0.185954
dtype: float64
True


In [4]:
spec = build_feature_spec(X_tr)
out = apply_feature_spec(X, spec)
print(out.shape)

(1567, 442)
